### Lab Assignment: Commercial Data Analysis

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: February 15, 2026

---

### INSTRUCTIONS  
In this assignment, you will work with a dataset containing information about businesses.  
Each record is a business location.  Follow the steps below, writing and running the code in blocks, and displaying the solutions.  

The path to the dataset is in a file named `find_dataset_on_rivanna.txt` in Module 3. 

Each question part is worth 1 POINT, for a total of 15 POINTS.

Hint: reaching deeper fields in json hierarchy can be done like this:  

`df.select('address.street_number')`

---

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
        .appName("comm") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 20:36:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/10 20:36:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/10 20:36:24 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [1]:
# note that read.json can read a zipped JSON directly

**1. (1 PT) Read in the dataset and show the number of records**

In [98]:
# This will read the data into a Spark DataFrame.
df_lab2 = spark.read.json("/standard/ds7200-apt4c/large_datasets/part-00000-a159c41a-bc58-4476-9b78-c437667f9c2b-c000.json.gz")

In [99]:
# Show the record count.
df_lab2.count()

154679

In [18]:
# Look at the structure to help with queries.
df_lab2.printSchema()

root
 |-- address: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- coordinates: struct (nullable = true)
 |    |    |-- lat: double (nullable = true)
 |    |    |-- lon: double (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- county: string (nullable = true)
 |    |-- full_address: string (nullable = true)
 |    |-- highway_number: string (nullable = true)
 |    |-- is_headquarters: boolean (nullable = true)
 |    |-- is_parsed: boolean (nullable = true)
 |    |-- post_direction: string (nullable = true)
 |    |-- pre_direction: string (nullable = true)
 |    |-- secondary_number: string (nullable = true)
 |    |-- state: string (nullable = true)
 |    |-- street: string (nullable = true)
 |    |-- street_address: string (nullable = true)
 |    |-- street_number: string (nullable = true)
 |    |-- street_type: string (nullable = true)
 |    |-- type_of_address: string (nullable = true)
 |    |-- zip: string (nullable = true)
 |    |-- 

**2. (1 PT) Show the first 3 records**

In [4]:
df_lab2.show(3)

+--------------------+--------------------+--------------------+----------------+----+--------------------+--------------------+--------------------+
|             address|       business_tags|               hours|              id|menu|             reviews|                urls|             webpage|
+--------------------+--------------------+--------------------+----------------+----+--------------------+--------------------+--------------------+
|{Woodburn, {45.15...|                NULL|                NULL|000023995a540868|NULL|                  []|{woodburn.k12.or....|{Educational Tech...|
|{Hialeah, {25.884...|{[], [{has_atm, Y...|{NULL, 1900, NULL...|0000821a1394916e|NULL|                NULL|{NULL, [yelp.com]...|                NULL|
|{Rochester, {43.1...|{[], [{accepts_cr...|{NULL, 1700, NULL...|000136e65d50c3b7|NULL|[{New (to me) qui...|{usps.com, [yelp....|{Welcome | USPS G...|
+--------------------+--------------------+--------------------+----------------+----+--------------

**3. (1 PT) Show the first 5 street addresses which are not null**  

In [11]:
# Register the DF as a temp view first.
df_lab2.createOrReplaceTempView("df_lab2")

In [59]:
# Write the query and show the results.
first_five = spark.sql("""
SELECT address.street_address 
FROM df_lab2 
WHERE address.street_address IS NOT NULL 
LIMIT 5
""").show()

+-------------------+
|     street_address|
+-------------------+
| Cooper Contracting|
|          Route 607|
|Bush St & Kearny St|
|          S 14th St|
|             Rr 474|
+-------------------+



**4. (1 PT) Location**  

Count the number of records where the city is New York

In [30]:
# Experimenting with carriage returns.
num_NYC = spark.sql("""
SELECT COUNT(*) AS NYC_cnt
FROM df_lab2 
WHERE address.city = 'New York'
""").show()

[Stage 14:>                                                         (0 + 1) / 1]

+-------+
|NYC_cnt|
+-------+
|   1953|
+-------+



**5. (1 PT) Hours**  

Count the number of records where closing time on Tuesday is 8pm

In [33]:
Tue8pm_close = spark.sql("""
SELECT COUNT(*) as Tue8pm_close_cnt
FROM df_lab2
WHERE hours.tuesday_close = '2000'
""").show()

[Stage 17:>                                                         (0 + 1) / 1]

+----------------+
|Tue8pm_close_cnt|
+----------------+
|            3154|
+----------------+



**6. (1 PT) Location and Hours**  

For the records where the city is New York, aggregate by Tuesday closing time, showing a column called `count` with the number of records for each Tuesday closing time. Sort by the `count` column in descending order. Here is an example of results output:

+-------------+-----+  
|tuesday_close|count|  
+-------------+-----+  
|         1800| 2001|  
|         1700| 1800|  
|         2000| 1550|  

In [35]:
NYC_Tue8pm_close = spark.sql("""
SELECT 
    hours.tuesday_close
    ,COUNT(*) as count
FROM
    df_lab2
WHERE
    address.city = 'New York'
    AND hours.tuesday_close IS NOT NULL
GROUP BY
    hours.tuesday_close
ORDER BY
    count DESC
""").show()

[Stage 23:>                                                         (0 + 1) / 1]

+-------------+-----+
|tuesday_close|count|
+-------------+-----+
|         1700|  174|
|         1800|   92|
|         1900|   57|
|         2000|   48|
|         2100|   35|
|         2300|   19|
|         2200|   18|
|         0000|   15|
|         1830|   14|
|         1730|   12|
|         1600|   10|
|         1930|    8|
|         2030|    7|
|         0100|    5|
|         2330|    5|
|         2359|    5|
|         0400|    5|
|         2230|    5|
|         1500|    4|
|         1630|    4|
+-------------+-----+
only showing top 20 rows


**7. (1 PT) Price Range**  

Price range is quoted in number of dollar signs.  Count the number of records with price range greater than or equal to three.

In [38]:
# First, let's see what the price_range values look like.
price_distinct = spark.sql("""
SELECT DISTINCT
    menu.price_range
FROM
    df_lab2
""").show()

[Stage 30:>                                                         (0 + 1) / 1]

+-----------+
|price_range|
+-----------+
|          3|
|          1|
|          4|
|          2|
|       NULL|
+-----------+



In [89]:
# Now, I'll get where this is greater than 3.
price_gt3 = spark.sql("""
SELECT DISTINCT
    COUNT(*) AS price_gt3_cnt
FROM
    df_lab2
WHERE
    menu.price_range > 3
""").show()

[Stage 104:>                                                        (0 + 1) / 1]

+-------------+
|price_gt3_cnt|
+-------------+
|           34|
+-------------+



**8. (1 PT) Missing Webpage URL**  

Count the number of records that are missing the webpage url.

In [88]:
url_null = spark.sql("""
SELECT
    COUNT(*) AS url_null_cnt
FROM
    df_lab2
WHERE
    webpage.url IS NULL
""").show()

[Stage 101:>                                                        (0 + 1) / 1]

+------------+
|url_null_cnt|
+------------+
|       79813|
+------------+



**9. (1 PT) Webpage URLs**  

Register the dataframe as a temp table.  
Next, use Spark SQL to select only the webpage title column, filtering on rows where the webpage url (accessed under `webpage.url`) is *Target.com*. 

Show only one resulting row and don't truncate the output.

In [56]:
url_Target = spark.sql("""
SELECT DISTINCT
    webpage.title
FROM
    df_lab2
WHERE
    webpage.url = 'Target.com'
""").show(truncate=False)

[Stage 71:>                                                         (0 + 1) / 1]

+-------------------------------+
|title                          |
+-------------------------------+
|Target : Expect More. Pay Less.|
+-------------------------------+



**10. (1 PT) Analysis on Ratings**  

The reviews contains information such as the number of stars for each review (the *rating*).  
The ratings are stored in an array (`reviews.stars`) for each business location (you should check for yourself). Return the top five most common rating arrays.  For example, an array might look like: 
[5, 5]



In [57]:
# Look at the values first.
stars_vals = spark.sql("""
SELECT DISTINCT reviews.stars
FROM df_lab2
LIMIT 5
""").show()

[Stage 74:>                                                         (0 + 1) / 1]

+--------------------+
|               stars|
+--------------------+
|[1, 5, 1, 5, 5, 1...|
|[3, 2, 5, 5, 5, 4...|
|[NULL, NULL, NULL...|
|[4, 5, 5, 1, NULL...|
|[NULL, NULL, NULL...|
+--------------------+



In [70]:
# Retrieve the desired results.
top5_ratings = spark.sql("""
SELECT
    reviews.stars AS top5_ratings,
    COUNT(*) AS top5_ratings_cnt
FROM 
    df_lab2
GROUP BY 
    reviews.stars
ORDER BY 
    top5_ratings_cnt DESC
LIMIT 5
""").show(truncate=False)

# The results show a lot of missing values.  
# These could be filtered out but I left as is...I hope this is sufficient for this question.

[Stage 82:>                                                         (0 + 1) / 1]

+------------+----------------+
|top5_ratings|top5_ratings_cnt|
+------------+----------------+
|NULL        |74679           |
|[]          |42419           |
|[5]         |4258            |
|[NULL]      |3067            |
|[5, 5]      |1610            |
+------------+----------------+



**11. More work with Ratings**  

For this question, you will filter out null ratings and then compute the average rating for each business location (using the field: `id`).


a) (1 PT) Create a new dataframe retaining two fields: `id`, `reviews.stars`


In [97]:
df_q11 = spark.sql("SELECT id, reviews.stars FROM df_lab2")
df_q11.show()

+----------------+--------------------+
|              id|               stars|
+----------------+--------------------+
|000023995a540868|                  []|
|0000821a1394916e|                NULL|
|000136e65d50c3b7|              [4, 4]|
|00014329a70b9869|                NULL|
|00031c0a83f00657|                NULL|
|0003b7589a4e12a0|                 [5]|
|00045f958e4bb02a|[NULL, NULL, NULL...|
|00055eb14ea38ca2|                NULL|
|00059519f0dba1b4|[NULL, NULL, NULL...|
|0005f8f0b2beeac4|                NULL|
|000673b2dac411ea|                NULL|
|0006d5aa170bae22|                  []|
|00080a1e9b3607c0|                NULL|
|0008bc70f8ba62bf|              [NULL]|
|0008c39401d7ce3b|                NULL|
|0009cd9a574331a9|                NULL|
|000a1df4c8e0ecd2|[NULL, NULL, 4, 5...|
|000a863f1d02dd78|                NULL|
|000b4a9059161ee7|                NULL|
|000bf1e934ac9cb6|                  []|
+----------------+--------------------+
only showing top 20 rows


b) (1 PT) Create a row for each rating  
hint: use the `withColumn()` and `explode()` functions  
you will need to import the `explode()` function by issuing:

`from pyspark.sql.functions import explode`


In [79]:
# Import explode.
from pyspark.sql.functions import explode

In [100]:
# Create the new column from the pivoted ratings values.
df_q11 = df_q11.withColumn('rating', explode("stars"))
df_q11.show()

+----------------+--------------------+------+
|              id|               stars|rating|
+----------------+--------------------+------+
|000136e65d50c3b7|              [4, 4]|     4|
|000136e65d50c3b7|              [4, 4]|     4|
|0003b7589a4e12a0|                 [5]|     5|
|00045f958e4bb02a|[NULL, NULL, NULL...|  NULL|
|00045f958e4bb02a|[NULL, NULL, NULL...|  NULL|
|00045f958e4bb02a|[NULL, NULL, NULL...|  NULL|
|00045f958e4bb02a|[NULL, NULL, NULL...|  NULL|
|00059519f0dba1b4|[NULL, NULL, NULL...|  NULL|
|00059519f0dba1b4|[NULL, NULL, NULL...|  NULL|
|00059519f0dba1b4|[NULL, NULL, NULL...|  NULL|
|00059519f0dba1b4|[NULL, NULL, NULL...|  NULL|
|00059519f0dba1b4|[NULL, NULL, NULL...|  NULL|
|00059519f0dba1b4|[NULL, NULL, NULL...|  NULL|
|00059519f0dba1b4|[NULL, NULL, NULL...|  NULL|
|00059519f0dba1b4|[NULL, NULL, NULL...|  NULL|
|00059519f0dba1b4|[NULL, NULL, NULL...|     1|
|00059519f0dba1b4|[NULL, NULL, NULL...|     5|
|00059519f0dba1b4|[NULL, NULL, NULL...|     2|
|00059519f0db

c) (1 PT) Return a count of the number of ratings in this dataframe

In [107]:
# First, just get the total count.
df_q11.count()

600082

In [101]:
# Then, try doing this without SQL but using PySpark commands, which are not natural for me.  
# In fact, I was so bad at this that I had to get help from an LLM to write this correctly.  :(
df_q11.groupBy("rating").count().orderBy("count", ascending=False).show()

# I intentionally broke this out by rating value to see how many are NULL.

[Stage 127:>                                                        (0 + 1) / 1]

+------+------+
|rating| count|
+------+------+
|     5|259351|
|     4|107728|
|     1| 80134|
|  NULL| 61841|
|     3| 51733|
|     2| 39295|
+------+------+



In [108]:
# Doing this with SQL to prove the above is correct.  This feels natural to me.  :)
df_q11.createOrReplaceTempView("df_q11")

rating_counts = spark.sql("""
SELECT
    rating
    ,COUNT(*) AS rating_cnt
    --,COUNT(rating) as rating_cnt <-NOTE: I first did this, then realizes this doesn't count NULLs.
FROM
    df_q11
GROUP BY
    rating
ORDER BY 
    rating_cnt DESC
""").show()

[Stage 148:>                                                        (0 + 1) / 1]

+------+----------+
|rating|rating_cnt|
+------+----------+
|     5|    259351|
|     4|    107728|
|     1|     80134|
|  NULL|     61841|
|     3|     51733|
|     2|     39295|
+------+----------+



d) (1 PT) Drop rows where the rating is null, and return a count of the number of non-null ratings

In [103]:
df_q11_d = spark.sql("SELECT * FROM df_q11 WHERE rating IS NOT NULL")
df_q11_d.count()

538241

e) (1 PT) Compute the average rating, grouped by `id`. After the average is computed, sort by `id` in ascending order and show the top 10 records.  
 
hint:   
this can all be done in one line using the `agg()` function  
this `id` should be at the top: 000136e65d50c3b7

In [128]:
# Import functions.
from pyspark.sql import functions as F
from pyspark.sql.functions import asc

In [129]:
# Using PySpark, which I don't like.  
agg_df_q11_d = df_q11_d.groupBy("id").agg(F.avg("rating")).sort(asc("id"))
agg_df_q11_d.show(10)

[Stage 166:>                                                        (0 + 1) / 1]

+----------------+------------------+
|              id|       avg(rating)|
+----------------+------------------+
|000136e65d50c3b7|               4.0|
|0003b7589a4e12a0|               5.0|
|00059519f0dba1b4|3.3333333333333335|
|000a1df4c8e0ecd2|               4.6|
|000c7b7a30623083|               5.0|
|000c9ffc8b89af03|               3.0|
|000de20baa847ecc|1.6666666666666667|
|001064359d9f162f|               5.0|
|0010c9f495d87dd7|               3.0|
|0017774db5e6400a| 4.333333333333333|
+----------------+------------------+
only showing top 10 rows
